# 08a — wav2vec2-XLS-R Learning Curves
**Model:** `facebook/wav2vec2-xls-r-300m`  |  **Source:** mscil  |  **Fractions:** 0.10, 0.25, 0.50

> 1.00 fraction already done (NB05b) — stub JSON pre-populated in `./results/`

In [ ]:
!pip install -q "transformers>=4.46" "datasets<3.0" "accelerate>=0.26" evaluate jiwer soundfile librosa pyctcdecode kenlm matplotlib
print("Done.")

In [ ]:
import os, gc, json, random, logging, subprocess, datetime
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Union, Optional

import numpy as np
import torch
import evaluate
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
from datasets import load_from_disk, load_dataset, Audio
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, TrainingArguments, Trainer, EarlyStoppingCallback

logging.getLogger("transformers").setLevel(logging.WARNING)

assert torch.cuda.is_available(), "No GPU found."
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)} ({vram_gb:.0f} GB)")

MODEL_KEY  = "wav2vec2-xlsr"
MODEL_ID   = "facebook/wav2vec2-xls-r-300m"
SOURCE     = "mscil"
FRACTIONS  = [0.10, 0.25, 0.50]   # 1.00 already done — stub in ./results/
DATASET_PATH = "./telugu_asr_clean_dataset"
PROC_PATH    = "./telugu_wav2vec2_processor_v2" if Path("./telugu_wav2vec2_processor_v2").exists() else "./telugu_wav2vec2_processor"
VOCAB_JSON   = "./vocab_v2.json" if Path("./vocab_v2.json").exists() else "./vocab.json"
KENLM_BIN    = "./kenlm/build/bin/lmplz"
RESULTS_DIR  = Path("./results")
RESULTS_DIR.mkdir(exist_ok=True)
SEED = 42

LR, WARMUP, EPOCHS      = 1e-4, 1000, 50
PATIENCE, THRESHOLD      = 8, 0.005
MASK_TIME, MASK_FEAT     = 0.1, 0.1
ATTN_DROP, HIDDEN_DROP   = 0.1, 0.1
EVAL_STEPS               = 500
BATCH, GRAD_ACCUM        = 8, 4

print(f"Fractions to run: {FRACTIONS}")
for frac in FRACTIONS:
    p = RESULTS_DIR / f"{MODEL_KEY}_lc_{SOURCE}_{frac:.2f}.json"
    print(f"  {frac}: {'DONE' if p.exists() else 'pending'}")

In [ ]:
fleurs_test = None
try:
    _fl = load_dataset("google/fleurs", "te_in", trust_remote_code=True)
    fleurs_test = _fl["test"].select_columns(["audio", "transcription"])
    fleurs_test = fleurs_test.cast_column("audio", Audio(sampling_rate=16_000))
    print(f"FLEURS te_in: {len(fleurs_test):,} samples")
except Exception as e:
    print(f"[WARN] FLEURS unavailable: {e}")

In [ ]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True
    def __call__(self, features):
        input_features = [{"input_values": f["input_values"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, padding=self.padding, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, padding=self.padding, return_tensors="pt")
        batch["labels"] = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        return batch

def load_split(fraction):
    ds = load_from_disk(DATASET_PATH).cast_column("audio", Audio(sampling_rate=16_000))
    if fraction < 1.0:
        ds = ds.shuffle(seed=SEED).select(range(int(len(ds) * fraction)))
    split = ds.train_test_split(test_size=0.1, seed=SEED)
    return split["train"], split["test"]

def prepare(train_ds, val_ds, processor):
    def _prep(batch):
        audio = batch["audio"]
        batch["input_values"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
        batch["input_length"] = len(batch["input_values"])
        batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
        return batch
    KEEP = {"input_values", "input_length", "labels"}
    tr = train_ds.map(_prep, remove_columns=[c for c in train_ds.column_names if c not in KEEP], num_proc=4, desc="train").sort("input_length")
    va = val_ds.map(_prep, remove_columns=[c for c in val_ds.column_names if c not in KEEP], num_proc=4, desc="val")
    return tr, va

def make_compute_metrics(processor):
    def compute_metrics(pred):
        pred_ids = np.argmax(pred.predictions, axis=-1)
        label_ids = pred.label_ids.copy()
        label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
        return {
            "wer": wer_metric.compute(predictions=processor.batch_decode(pred_ids), references=processor.batch_decode(label_ids, group_tokens=False)),
            "cer": cer_metric.compute(predictions=processor.batch_decode(pred_ids), references=processor.batch_decode(label_ids, group_tokens=False)),
        }
    return compute_metrics

def decode_greedy_fleurs(model, processor):
    if fleurs_test is None: return float("nan")
    device = next(model.parameters()).device
    model.eval()
    preds, refs = [], []
    for i, s in enumerate(fleurs_test):
        inp = processor(s["audio"]["array"], sampling_rate=16_000, return_tensors="pt", padding=True)
        with torch.no_grad():
            logits = model(input_values=inp.input_values.to(device)).logits
        preds.append(processor.batch_decode(torch.argmax(logits, dim=-1))[0])
        refs.append(s["transcription"])
        if (i+1) % 200 == 0: print(f"  FLEURS greedy: {i+1}/{len(fleurs_test)}")
    return wer_metric.compute(predictions=preds, references=refs)

def build_arpa(train_ds, tag):
    if not Path(KENLM_BIN).exists():
        print("[WARN] KenLM binary not found — skipping LM")
        return None
    txt = f"./telugu_lm_{tag}.txt"
    arpa = f"./telugu_kenlm_{tag}.arpa"
    with open(txt, "w") as f: f.write("\n".join(train_ds["transcription"]))
    with open(txt, "rb") as fin, open(arpa, "w") as fout:
        result = subprocess.run([KENLM_BIN, "-o", "5", "--discount_fallback"], stdin=fin, stdout=fout)
    if result.returncode != 0:
        print(f"[WARN] KenLM lmplz failed (exit {result.returncode}) — skipping LM decoding")
        return None
    return arpa

def decode_kenlm(model, processor, dataset, arpa):
    import pyctcdecode
    vocab = sorted(json.load(open(VOCAB_JSON)).items(), key=lambda x: x[1])
    decoder = pyctcdecode.build_ctcdecoder([k for k,_ in vocab], arpa, alpha=0.5, beta=1.0)
    device = next(model.parameters()).device
    model.eval()
    preds, refs = [], []
    for i, s in enumerate(dataset):
        inp = processor(s["audio"]["array"], sampling_rate=16_000, return_tensors="pt", padding=True)
        with torch.no_grad():
            logits = model(input_values=inp.input_values.to(device)).logits[0].cpu().numpy()
        preds.append(decoder.decode(logits))
        refs.append(s["transcription"])
        if (i+1) % 500 == 0: print(f"  KenLM: {i+1}/{len(dataset)}")
    return wer_metric.compute(predictions=preds, references=refs)

def save_result(d, frac):
    p = RESULTS_DIR / f"{MODEL_KEY}_lc_{SOURCE}_{frac:.2f}.json"
    with open(p, "w") as f: json.dump(d, f, indent=2)
    with open(RESULTS_DIR / "run_log.txt", "a") as f:
        f.write(f"{datetime.datetime.now().isoformat()} | {d['run_tag']} | greedy={d.get('greedy_wer_val')} | kenlm_fleurs={d.get('kenlm_wer_fleurs')}\n")
    print(f"Saved: {p.name}")

def update_plot():
    try:
        rows = [json.load(open(p)) for p in sorted(RESULTS_DIR.glob(f"{MODEL_KEY}_lc_*.json"))]
        if not rows: return
        rows.sort(key=lambda r: r.get("n_train_samples", 0))
        fig, ax = plt.subplots(figsize=(9, 5))
        xs = [r["n_train_samples"] for r in rows]
        ax.plot(xs, [r.get("greedy_wer_val", 0)*100 for r in rows], "o--", label="greedy WER (val)", color="steelblue")
        kenlm = [r.get("kenlm_wer_val") for r in rows]
        if any(v for v in kenlm):
            ax.plot(xs, [v*100 if v else None for v in kenlm], "o-", label="KenLM WER (val)", color="darkorange")
        fleurs = [r.get("kenlm_wer_fleurs") or r.get("greedy_wer_fleurs") for r in rows]
        if any(v for v in fleurs):
            ax.plot(xs, [v*100 if v else None for v in fleurs], "*-", label="FLEURS WER", color="green", markersize=12)
        ax.set_xscale("log"); ax.set_xlabel("Training samples"); ax.set_ylabel("WER %")
        ax.set_title("wav2vec2-XLS-R Learning Curve"); ax.legend(); ax.grid(True, alpha=0.3)
        plt.tight_layout(); fig.savefig(RESULTS_DIR / "08a_wav2vec2_curve.png", dpi=150)
        clear_output(wait=True); display(fig); plt.close(fig)
    except Exception as e:
        print(f"[WARN] Plot failed: {e}")

def _safe(v):
    try: return None if (v is None or np.isnan(v)) else round(float(v), 4)
    except: return None

print("Helpers ready.")

In [ ]:
processor = Wav2Vec2Processor.from_pretrained(PROC_PATH)

for frac in FRACTIONS:
    result_path = RESULTS_DIR / f"{MODEL_KEY}_lc_{SOURCE}_{frac:.2f}.json"
    if result_path.exists():
        print(f"SKIP {frac}: already done")
        continue

    print(f"\n{'='*50}\nSTART frac={frac}\n{'='*50}")
    run_tag = f"{MODEL_KEY}_{SOURCE}_{frac:.2f}"
    output_dir = f"./wav2vec2-lc-{run_tag}"

    train_ds, val_ds = load_split(frac)
    n_train = len(train_ds)
    approx_hours = round(n_train * 3.85 / 3600, 1)
    print(f"Train: {n_train:,}  Val: {len(val_ds):,}  ~{approx_hours}h")

    train_prep, val_prep = prepare(train_ds, val_ds, processor)

    model = Wav2Vec2ForCTC.from_pretrained(
        MODEL_ID, attention_dropout=ATTN_DROP, hidden_dropout=HIDDEN_DROP,
        feat_proj_dropout=0.0, mask_time_prob=MASK_TIME, mask_time_length=10,
        mask_feature_prob=MASK_FEAT, ctc_loss_reduction="mean", ctc_zero_infinity=True,
        pad_token_id=processor.tokenizer.pad_token_id,
        vocab_size=processor.tokenizer.vocab_size, ignore_mismatched_sizes=True,
    )
    model.freeze_feature_encoder()
    model.gradient_checkpointing_enable()

    args = TrainingArguments(
        output_dir=output_dir, num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LR,
        warmup_steps=WARMUP, lr_scheduler_type="cosine", fp16=True,
        dataloader_num_workers=4, eval_strategy="steps", eval_steps=EVAL_STEPS,
        save_strategy="steps", save_steps=EVAL_STEPS, save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model="wer",
        greater_is_better=False, logging_steps=100, report_to="none",
        seed=SEED, remove_unused_columns=False, label_names=["labels"],
    )
    trainer = Trainer(
        model=model, args=args, train_dataset=train_prep, eval_dataset=val_prep,
        processing_class=processor.feature_extractor,
        data_collator=DataCollatorCTCWithPadding(processor=processor),
        compute_metrics=make_compute_metrics(processor),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE, early_stopping_threshold=THRESHOLD)],
    )
    ckpts = sorted(Path(output_dir).glob("checkpoint-*"))
    resume = str(ckpts[-1]) if ckpts else None
    if resume: print(f"Resuming from {resume}")
    trainer.train(resume_from_checkpoint=resume)
    trainer.save_model(f"{output_dir}/final")
    processor.save_pretrained(f"{output_dir}/final")

    val_m = trainer.evaluate()
    greedy_wer = val_m.get("eval_wer", float("nan"))
    greedy_cer = val_m.get("eval_cer", float("nan"))
    epochs_done = int(trainer.state.epoch or 0)
    print(f"Greedy WER (val): {greedy_wer:.4f}")

    greedy_wer_fleurs = decode_greedy_fleurs(model, processor)
    print(f"Greedy WER (FLEURS): {greedy_wer_fleurs:.4f}")

    arpa = build_arpa(train_ds, run_tag)
    kenlm_wer_val = decode_kenlm(model, processor, val_ds, arpa) if arpa else float("nan")
    kenlm_wer_fleurs = decode_kenlm(model, processor, fleurs_test, arpa) if (arpa and fleurs_test is not None) else float("nan")
    print(f"KenLM WER (val): {kenlm_wer_val:.4f}  FLEURS: {kenlm_wer_fleurs:.4f}")

    save_result({
        "run_tag": run_tag, "model_key": MODEL_KEY, "model_id": MODEL_ID,
        "arch": "ctc", "dataset_source": SOURCE, "data_fraction": frac,
        "n_train_samples": n_train, "approx_hours": approx_hours,
        "greedy_wer_val": _safe(greedy_wer), "greedy_cer_val": _safe(greedy_cer),
        "kenlm_wer_val": _safe(kenlm_wer_val),
        "greedy_wer_fleurs": _safe(greedy_wer_fleurs),
        "kenlm_wer_fleurs": _safe(kenlm_wer_fleurs),
        "beam_wer_val": None, "beam_wer_fleurs": None,
        "lr": LR, "effective_batch": BATCH * GRAD_ACCUM,
        "epochs_trained": epochs_done, "timestamp": datetime.datetime.now().isoformat(),
    }, frac)
    update_plot()

    del model; torch.cuda.empty_cache(); gc.collect()
    print("VRAM freed.")

print("\nAll wav2vec2 runs complete.")
update_plot()

In [ ]:
import pandas as pd
rows = [json.load(open(p)) for p in sorted(RESULTS_DIR.glob(f"{MODEL_KEY}_lc_*.json"))]
df = pd.DataFrame(rows)[["data_fraction", "n_train_samples", "greedy_wer_val", "kenlm_wer_val", "kenlm_wer_fleurs", "epochs_trained"]]
display(df.sort_values("data_fraction").reset_index(drop=True))